# scGPT Fine-tuning for MAPT Tangle Stage Prediction

**Input:** `neurons_hvg_preprocessed.h5ad` (from `scgpt_preprocessing.ipynb`)
**Model:** scGPT whole-human (pretrained on 33M human cells)
**Task:** 4-class classification â€” Tangle Stage 1, 2, 5, 6 â€” **evaluated on held-out donors**

---
**Pipeline:**
1. Setup (fold / seed / sequence length from environment variables, so one notebook serves a SLURM array)
2. Load data and model config
3. **Donor-level split** â€” test donors never appear in training
4. Tokenize with **scGPT's own preprocessing** (per-cell binning, `<cls>` token, correct padding)
5. DataLoaders (one imbalance correction: weighted sampler)
6. Load pretrained scGPT
7. **Baseline: linear probe on frozen scGPT embeddings**
8. Fine-tune last 2 transformer layers + classifier
9. Evaluate on test donors (balanced accuracy, macro-F1, per-class recall, donor-level accuracy)
10. Embedding visualization (test donors only) and save results

**Changes from v1:** pretrained attention weights now actually load (v1 silently left all 12 attention layers random because of a key-name mismatch); donor-level split (v1 split cells randomly, leaking donors into test); tokenization via `scgpt.preprocess.Preprocessor` + `tokenize_and_pad_batch` (v1 used z-scored input, no `<cls>` token, unmasked padding and per-gene binning); single imbalance correction (v1 stacked sampler + class weights + a manual 3x Stage 6 weight); baselines and imbalance-aware metrics.

Run one fold interactively, or all folds with `sbatch slurm/02_finetune_array.sbatch`.

## 1. Setup

In [1]:
import os
import sys
import json
import time
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
from scipy.sparse import issparse
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import umap

PROJECT_DIR = os.environ.get('SCGPT_PROJECT_DIR', os.getcwd())
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
from splits import make_donor_folds, split_indices, describe_split
from metrics_utils import compute_metrics, majority_baseline, STAGE_NAMES

# Run configuration (override with environment variables, e.g. from SLURM)
FOLD = int(os.environ.get('FOLD', 0))            # which donor fold is the test set
SEED = int(os.environ.get('SEED', 42))           # training seed (fold assignment is fixed separately)
MAX_LEN = int(os.environ.get('MAX_LEN', 513))    # genes per cell incl. <cls>
N_EPOCHS = int(os.environ.get('N_EPOCHS', 15))
PATIENCE = int(os.environ.get('PATIENCE', 3))
BATCH_SIZE = int(os.environ.get('BATCH_SIZE', 32))
LR = float(os.environ.get('LR', 1e-4))
UNFREEZE_LAST_N = int(os.environ.get('UNFREEZE_LAST_N', 2))
RUN_TAG = f'fold{FOLD}_seed{SEED}'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for d in ['figures', 'results', 'checkpoints']:
    os.makedirs(d, exist_ok=True)

print(f'Run: {RUN_TAG} | MAX_LEN={MAX_LEN} | epochs<={N_EPOCHS} | batch={BATCH_SIZE} | lr={LR}')
print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

Run: fold2_seed7 | MAX_LEN=513 | epochs<=15 | batch=32 | lr=0.0001
Device: cuda | GPU: Tesla V100-SXM2-32GB


## 2. Load Preprocessed Data and Model Configuration

In [2]:
MODEL_DIR = './scgpt_human_model'
PAD_TOKEN = '<pad>'
CLS_TOKEN = '<cls>'

adata = ad.read_h5ad('neurons_hvg_preprocessed.h5ad')
with open(f'{MODEL_DIR}/args.json') as f:
    model_args = json.load(f)
with open(f'{MODEL_DIR}/vocab.json') as f:
    vocab = json.load(f)

PAD_VALUE = model_args.get('pad_value', -2)
assert 'donor_id' in adata.obs, 'donor_id missing - rerun scgpt_preprocessing.ipynb (fixed version)'
X_min = adata.X.min()
assert X_min >= 0, f'X min is {X_min:.2f}: data looks z-scored - rerun scgpt_preprocessing.ipynb (fixed version)'

print(f'Cells: {adata.n_obs:,} | Genes: {adata.n_vars:,} | Donors: {adata.obs["donor_id"].nunique()}')
print(f'Vocab: {len(vocab):,} | <pad>={vocab[PAD_TOKEN]} | <cls>={vocab[CLS_TOKEN]} | pad_value={PAD_VALUE}')
print(f'Model: emb={model_args["embsize"]}, layers={model_args["nlayers"]}, heads={model_args["nheads"]}, bins={model_args["n_bins"]}')

Cells: 12,331 | Genes: 3,001 | Donors: 18
Vocab: 60,697 | <pad>=60694 | <cls>=60695 | pad_value=-2
Model: emb=512, layers=12, heads=8, bins=51


## 3. Donor-Level Split

Donors (not cells) are assigned to folds, stratified by stage (`splits.py`). For fold `FOLD`: test = that fold, validation = the next fold, training = the rest. The same folds are used by `baselines.py`, so all models are compared on identical test donors.

In [3]:
obs = adata.obs
labels_all = obs['tangle_label'].to_numpy().astype(int)
donors_all = obs['donor_id'].astype(str).to_numpy()

folds, n_splits = make_donor_folds(obs)
assert FOLD < n_splits, f'FOLD={FOLD} but only {n_splits} folds are possible'
train_idx, val_idx, test_idx = split_indices(obs, folds, n_splits, FOLD)

# No donor may appear in more than one split
tr_d, va_d, te_d = (set(donors_all[i]) for i in (train_idx, val_idx, test_idx))
assert not (tr_d & va_d or tr_d & te_d or va_d & te_d), 'Donor leakage between splits!'

print(f'{n_splits} folds; running test fold {FOLD}')
print(f'Train: {len(train_idx):,} cells / {len(tr_d)} donors')
print(f'Val:   {len(val_idx):,} cells / {len(va_d)} donors')
print(f'Test:  {len(test_idx):,} cells / {len(te_d)} donors')
split_table = describe_split(obs, {'train': train_idx, 'val': val_idx, 'test': test_idx})
print(split_table)

3 folds; running test fold 2
Train: 3,523 cells / 6 donors
Val:   3,899 cells / 6 donors
Test:  4,909 cells / 6 donors
      n_donors           n_cells            
split     test train val    test train   val
label                                       
0            1     1   1    1033   275   610
1            2     1   1    1397    39    96
2            1     1   1     868   655   926
3            2     3   3    1611  2554  2267


## 4. Tokenization (scGPT's own pipeline)

v1 hand-rolled tokenization with four problems: it read **z-scored** data, binned **per gene across cells** (pretraining bins **per cell**), placed no **`<cls>`** token at position 0 (so the "cell embedding" was just the first gene), and padded with id 0 while masking `vocab['<pad>']` (padding was attended to).

Now:
1. Keep genes present in the scGPT vocabulary
2. `Preprocessor(binning=51)` on the log1p matrix â€” per-cell quantile binning, exactly as in pretraining. Binning uses each cell's own values only, so it cannot leak test information
3. `tokenize_and_pad_batch(append_cls=True)` â€” prepends `<cls>`, drops zero-expression genes, pads with `vocab['<pad>']` / `pad_value`; cells with more than `MAX_LEN-1` expressed genes are randomly subsampled (scGPT's default)

In [4]:
from scgpt.preprocess import Preprocessor
from scgpt.tokenizer import tokenize_and_pad_batch

in_vocab = np.array([g in vocab for g in adata.var_names])
adata = adata[:, in_vocab].copy()
print(f'Genes in scGPT vocab: {in_vocab.sum()} / {len(in_vocab)} | MAPT kept: {"MAPT" in adata.var_names}')

# scgpt 0.2.2 calls sparse_matrix.A, which SciPy >= 1.14 removed -> pass a dense
# matrix (~12k cells x ~2.3k genes float32 is ~110 MB)
if issparse(adata.X):
    adata.X = adata.X.toarray().astype(np.float32)

preprocessor = Preprocessor(
    use_key='X',                  # log1p-normalized expression
    filter_gene_by_counts=False,
    filter_cell_by_counts=False,
    normalize_total=False,        # already normalized in preprocessing notebook
    log1p=False,                  # already log1p
    subset_hvg=False,             # already HVG-selected
    binning=model_args['n_bins'],
    result_binned_key='X_binned',
)
preprocessor(adata, batch_key=None)

binned = adata.layers['X_binned']
binned = binned.toarray() if issparse(binned) else np.asarray(binned)
gene_ids = np.array([vocab[g] for g in adata.var_names], dtype=int)

n_expr = (binned > 0).sum(1)
print(f'Expressed genes per cell: median {int(np.median(n_expr))}, '
      f'{(n_expr > MAX_LEN - 1).mean()*100:.1f}% of cells exceed MAX_LEN-1={MAX_LEN-1} (randomly subsampled)')

tokenized = tokenize_and_pad_batch(
    binned, gene_ids,
    max_len=MAX_LEN,
    vocab=vocab,
    pad_token=PAD_TOKEN,
    pad_value=PAD_VALUE,
    append_cls=True,
    include_zero_gene=False,
    cls_token=CLS_TOKEN,
    return_pt=True,
)
all_gene_tokens = tokenized['genes']     # LongTensor (n_cells, MAX_LEN)
all_values = tokenized['values']         # FloatTensor (n_cells, MAX_LEN)

assert (all_gene_tokens[:, 0] == vocab[CLS_TOKEN]).all(), '<cls> must be at position 0'
print(f'Tokenized: {tuple(all_gene_tokens.shape)} | '
      f'padding fraction: {(all_gene_tokens == vocab[PAD_TOKEN]).float().mean():.2f}')

Genes in scGPT vocab: 2316 / 3001 | MAPT kept: True


scGPT - INFO - Binning data ...


Expressed genes per cell: median 339, 14.1% of cells exceed MAX_LEN-1=512 (randomly subsampled)


Tokenized: (12331, 513) | padding fraction: 0.36


## 5. Datasets and DataLoaders

Class imbalance (Stage 6 â‰ˆ 52% of cells) is handled by **one** mechanism: a `WeightedRandomSampler` that draws training batches with balanced classes. The loss is plain cross-entropy. (v1 stacked the sampler, inverse-frequency loss weights and a hand-tuned 3x Stage-6 weight, which over-corrected and was tuned after looking at results.)

In [5]:
class TangleStageDataset(Dataset):
    def __init__(self, idx):
        self.idx = np.asarray(idx)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        return {'gene_ids': all_gene_tokens[j], 'values': all_values[j],
                'label': int(labels_all[j]), 'cell': int(j)}


train_labels = labels_all[train_idx]
class_counts = np.bincount(train_labels, minlength=4)
sampler = WeightedRandomSampler(
    weights=torch.as_tensor((1.0 / np.maximum(class_counts, 1))[train_labels], dtype=torch.double),
    num_samples=len(train_idx), replacement=True,
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(TangleStageDataset(train_idx), batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(TangleStageDataset(val_idx), batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(TangleStageDataset(test_idx), batch_size=64, shuffle=False, num_workers=2)
all_loader = DataLoader(TangleStageDataset(np.arange(len(labels_all))), batch_size=64, shuffle=False, num_workers=2)

print(f'Training class counts: {class_counts.tolist()}')
print(f'Sample batch labels: {torch.bincount(next(iter(train_loader))["label"], minlength=4).tolist()} (â‰ˆ balanced)')

Training class counts: [275, 39, 655, 2554]


Sample batch labels: [11, 4, 4, 13] (â‰ˆ balanced)


## 6. Load Pretrained scGPT

Architecture is read from `args.json`. Pretrained weights are copied for every parameter whose name and shape match; only the new classification head (`cls_decoder`) is randomly initialized.

**v1 bug:** the checkpoint stores attention weights as `self_attn.Wqkv.*` (flash-attention naming) but the model, built with `use_fast_transformer=False`, expects `self_attn.in_proj_*`. With `strict=False` these were silently skipped â€” v1's "34 missing keys" = 24 attention tensors (12 layers Ã— weight/bias) + 10 classifier tensors â€” so every attention layer was random. The keys are now renamed, and an assertion fails if anything other than the classifier head is left uninitialized.

In [6]:
from scgpt.model import TransformerModel


def build_and_load_model(n_cls=4):
    model = TransformerModel(
        ntoken=len(vocab),
        d_model=model_args['embsize'],
        nhead=model_args['nheads'],
        d_hid=model_args['d_hid'],
        nlayers=model_args['nlayers'],
        nlayers_cls=3,
        n_cls=n_cls,
        vocab=vocab,
        dropout=model_args['dropout'],
        pad_token=PAD_TOKEN,
        pad_value=PAD_VALUE,
        do_mvc=False,
        do_dab=False,
        use_batch_labels=False,
        domain_spec_batchnorm=False,
        input_emb_style='continuous',
        n_input_bins=model_args['n_bins'],
        cell_emb_style='cls',
        ecs_threshold=0.0,
        explicit_zero_prob=False,
        use_fast_transformer=False,
        pre_norm=False,
    )
    pretrained = torch.load(f'{MODEL_DIR}/best_model.pt', map_location='cpu')
    # The checkpoint was trained with flash-attention, which names the fused
    # attention projection "self_attn.Wqkv.*". With use_fast_transformer=False
    # PyTorch expects "self_attn.in_proj_*". Without this rename (v1 bug) all
    # 12 attention layers silently stay RANDOM.
    pretrained = {k.replace('self_attn.Wqkv.', 'self_attn.in_proj_'): v for k, v in pretrained.items()}

    model_dict = model.state_dict()
    loadable = {k: v for k, v in pretrained.items()
                if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(loadable)
    model.load_state_dict(model_dict)

    not_loaded = [k for k in model_dict if k not in loadable]
    print(f'Loaded {len(loadable)} pretrained tensors; newly initialized: {len(not_loaded)}')
    print(f'  Newly initialized: {not_loaded}')
    # Everything except the new classification head must come from the checkpoint
    unexpected_new = [k for k in not_loaded if not k.startswith('cls_decoder')]
    assert not unexpected_new, f'Pretrained weights missing for: {unexpected_new}'
    return model.to(device)


def forward(model, batch):
    gene_ids = batch['gene_ids'].to(device)
    values = batch['values'].to(device).float()
    out = model(
        gene_ids, values,
        src_key_padding_mask=gene_ids.eq(vocab[PAD_TOKEN]),
        batch_labels=None,
        CLS=True, CCE=False, MVC=False, ECS=False, do_sample=False,
    )
    return out['cls_output'], out['cell_emb']


@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, embs, cells = [], [], []
    for batch in loader:
        logits, emb = forward(model, batch)
        probs.append(torch.softmax(logits.float(), 1).cpu().numpy())
        embs.append(emb.float().cpu().numpy())
        cells.append(batch['cell'].numpy())
    return np.concatenate(probs), np.concatenate(embs), np.concatenate(cells)


model = build_and_load_model()

Loaded 159 pretrained tensors; newly initialized: 10
  Newly initialized: ['cls_decoder._decoder.0.weight', 'cls_decoder._decoder.0.bias', 'cls_decoder._decoder.2.weight', 'cls_decoder._decoder.2.bias', 'cls_decoder._decoder.3.weight', 'cls_decoder._decoder.3.bias', 'cls_decoder._decoder.5.weight', 'cls_decoder._decoder.5.bias', 'cls_decoder.out_layer.weight', 'cls_decoder.out_layer.bias']


## 7. Baseline: Linear Probe on Frozen scGPT Embeddings

Before any fine-tuning, extract the pretrained `<cls>` embedding for every cell and fit a logistic regression on training donors. This answers: *does pretrained scGPT already encode tangle stage, and does fine-tuning add anything beyond that?* The majority-class baseline is computed on the same test donors. (`baselines.py` adds a PCA + logistic regression baseline on the same folds.)

In [7]:
_, frozen_emb, cell_order = predict(model, all_loader)
assert (cell_order == np.arange(len(labels_all))).all()

probe_train = np.concatenate([train_idx, val_idx])   # no early stopping needed for the probe
probe = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=2000, class_weight='balanced'))
probe.fit(frozen_emb[probe_train], labels_all[probe_train])
probe_probs = probe.predict_proba(frozen_emb[test_idx])

frozen_probe_metrics = compute_metrics(labels_all[test_idx], probe_probs.argmax(1),
                                       probe_probs, donors_all[test_idx])
majority_metrics = majority_baseline(labels_all[probe_train], labels_all[test_idx], donors_all[test_idx])

for name, m in [('Majority class', majority_metrics), ('Frozen scGPT + LR', frozen_probe_metrics)]:
    print(f"{name:<18} acc={m['accuracy']:.3f}  bal_acc={m['balanced_accuracy']:.3f}  "
          f"macro_f1={m['macro_f1']:.3f}  macro_auc={m['macro_auc']}")

Majority class     acc=0.328  bal_acc=0.250  macro_f1=0.124  macro_auc=0.5
Frozen scGPT + LR  acc=0.202  bal_acc=0.208  macro_f1=0.184  macro_auc=0.471979071915147


## 8. Fine-tuning

- **Trainable:** last `UNFREEZE_LAST_N` (default 2) transformer layers + classification head; earlier layers stay frozen to keep general geneâ€“gene knowledge and limit overfitting
- **Loss:** plain cross-entropy (the sampler already balances classes)
- **Optimizer:** AdamW, lr 1e-4, weight decay 0.01; gradient clipping at 1.0
- **Model selection / early stopping:** validation **macro-F1** on the validation donors (patience 3); LR halves when it plateaus

In [8]:
def configure_finetuning(model, unfreeze_last_n):
    n_layers = model_args['nlayers']
    for p in model.parameters():
        p.requires_grad = False
    names = ['cls_decoder'] + [f'transformer_encoder.layers.{n_layers - 1 - i}.' for i in range(unfreeze_last_n)]
    for n, p in model.named_parameters():
        if any(x in n for x in names):
            p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable:,} / {total:,} ({trainable/total*100:.1f}%) | unfrozen: {names}')
    return model


model = configure_finetuning(model, UNFREEZE_LAST_N)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
ckpt_path = f'checkpoints/scgpt_{RUN_TAG}.pt'

history = {'train_loss': [], 'val_loss': [], 'val_macro_f1': [], 'val_bal_acc': []}
best_f1, bad_epochs = -1.0, 0

for epoch in range(N_EPOCHS):
    t0 = time.time()
    model.train()
    running = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        logits, _ = forward(model, batch)
        loss = criterion(logits, batch['label'].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running += loss.item()

    val_probs, _, _ = predict(model, val_loader)
    y_val = labels_all[val_idx]
    val_loss = nn.functional.nll_loss(torch.log(torch.as_tensor(val_probs) + 1e-9),
                                      torch.as_tensor(y_val)).item()
    val_m = compute_metrics(y_val, val_probs.argmax(1))
    scheduler.step(val_m['macro_f1'])

    history['train_loss'].append(running / len(train_loader))
    history['val_loss'].append(val_loss)
    history['val_macro_f1'].append(val_m['macro_f1'])
    history['val_bal_acc'].append(val_m['balanced_accuracy'])
    print(f"Epoch {epoch+1:2d} | train loss {history['train_loss'][-1]:.4f} | val loss {val_loss:.4f} | "
          f"val macro-F1 {val_m['macro_f1']:.3f} | val bal-acc {val_m['balanced_accuracy']:.3f} | "
          f"pred dist {np.bincount(val_probs.argmax(1), minlength=4).tolist()} | {time.time()-t0:.0f}s")

    if val_m['macro_f1'] > best_f1:
        best_f1, bad_epochs = val_m['macro_f1'], 0
        torch.save(model.state_dict(), ckpt_path)
        print(f'  -> best model saved (val macro-F1 {best_f1:.3f})')
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}')
            break

Trainable: 3,685,380 / 51,333,637 (7.2%) | unfrozen: ['cls_decoder', 'transformer_encoder.layers.11.', 'transformer_encoder.layers.10.']


Epoch  1 | train loss 0.8430 | val loss 2.1320 | val macro-F1 0.214 | val bal-acc 0.297 | pred dist [1906, 201, 1168, 624] | 24s


  -> best model saved (val macro-F1 0.214)


Epoch  2 | train loss 0.6022 | val loss 1.8099 | val macro-F1 0.246 | val bal-acc 0.266 | pred dist [707, 370, 1137, 1685] | 24s


  -> best model saved (val macro-F1 0.246)


...

Epoch  7 | train loss 0.2988 | val loss 2.0430 | val macro-F1 0.308 | val bal-acc 0.322 | pred dist [890, 89, 1252, 1668] | 24s


  -> best model saved (val macro-F1 0.308)


Epoch  8 | train loss 0.2888 | val loss 2.3347 | val macro-F1 0.303 | val bal-acc 0.346 | pred dist [1257, 187, 874, 1581] | 24s


Epoch  9 | train loss 0.2577 | val loss 2.3468 | val macro-F1 0.282 | val bal-acc 0.300 | pred dist [926, 114, 1200, 1659] | 24s


Epoch 10 | train loss 0.2225 | val loss 2.2937 | val macro-F1 0.294 | val bal-acc 0.309 | pred dist [836, 123, 1335, 1605] | 24s
Early stopping at epoch 10


## 9. Evaluation on Held-out Test Donors

Headline metrics are **balanced accuracy** and **macro-F1** (not accuracy â€” Stage 6 dominates). **Donor-level accuracy** averages each test donor's cell probabilities and asks whether the donor's stage is recovered, which is the clinically meaningful unit.

In [9]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))
test_probs, test_emb, _ = predict(model, test_loader)
y_test = labels_all[test_idx]
finetuned_metrics = compute_metrics(y_test, test_probs.argmax(1), test_probs, donors_all[test_idx])

comparison = pd.DataFrame({
    name: {k: m[k] for k in ['accuracy', 'balanced_accuracy', 'macro_f1', 'macro_auc']}
          | {'donor_accuracy': m['donor_level']['accuracy']}
    for name, m in [('majority', majority_metrics),
                    ('frozen_scgpt_probe', frozen_probe_metrics),
                    ('finetuned_scgpt', finetuned_metrics)]
}).T.round(3)
print(f'=== Test fold {FOLD}: {len(test_idx):,} cells from {len(te_d)} donors ===')
print(comparison)
print('\nFine-tuned per-class recall:', {k: round(v, 3) for k, v in finetuned_metrics['per_class_recall'].items()})

cm = np.array(finetuned_metrics['confusion_matrix'])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=STAGE_NAMES, yticklabels=STAGE_NAMES, ax=ax)
ax.set_xlabel('Predicted stage')
ax.set_ylabel('True stage')
ax.set_title(f'Fine-tuned scGPT â€” held-out donors ({RUN_TAG})')
plt.tight_layout()
plt.savefig(f'figures/confusion_matrix_{RUN_TAG}.png', dpi=150)

=== Test fold 2: 4,909 cells from 6 donors ===
                    accuracy  balanced_accuracy  macro_f1  macro_auc  \
majority               0.328              0.250     0.124      0.500   
frozen_scgpt_probe     0.202              0.208     0.184      0.472   
finetuned_scgpt        0.270              0.275     0.210      0.535   

                    donor_accuracy  
majority                     0.333  
frozen_scgpt_probe           0.000  
finetuned_scgpt              0.500  

Fine-tuned per-class recall: {'Stage 1': 0.058, 'Stage 2': 0.005, 'Stage 5': 0.559, 'Stage 6': 0.479}


## 10. Embedding Visualization (test donors only)

v1 plotted all cells, most of which the model had trained on. Here only **held-out test donors** are shown, colored by stage, by donor and by prediction correctness. If cells group mainly by donor, the embedding still carries strong individual/batch signal.

In [10]:
reducer = umap.UMAP(n_neighbors=15, min_dist=0.3, random_state=SEED)
emb2d = reducer.fit_transform(test_emb)
correct = test_probs.argmax(1) == y_test
test_donors = donors_all[test_idx]

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
for c, name in enumerate(STAGE_NAMES):
    m = y_test == c
    axes[0].scatter(emb2d[m, 0], emb2d[m, 1], s=4, alpha=0.5, c=colors[c], label=name)
axes[0].legend(markerscale=4)
axes[0].set_title('True stage')
for d in np.unique(test_donors):
    m = test_donors == d
    axes[1].scatter(emb2d[m, 0], emb2d[m, 1], s=4, alpha=0.5, label=d)
axes[1].legend(markerscale=4, fontsize=7)
axes[1].set_title('Donor')
axes[2].scatter(emb2d[correct, 0], emb2d[correct, 1], s=4, alpha=0.5, c='#2196F3', label='correct')
axes[2].scatter(emb2d[~correct, 0], emb2d[~correct, 1], s=4, alpha=0.5, c='#F44336', label='incorrect')
axes[2].legend(markerscale=4)
axes[2].set_title(f'Prediction correctness (acc {correct.mean():.2f})')
plt.suptitle(f'Fine-tuned scGPT embeddings â€” held-out donors ({RUN_TAG})')
plt.tight_layout()
plt.savefig(f'figures/UMAP_test_donors_{RUN_TAG}.png', dpi=150, bbox_inches='tight')

## 11. Save Results

Results for this fold/seed are written to `results/scgpt_{RUN_TAG}.json`. After all folds finish, `python aggregate_results.py` combines them with `results/baselines.json` into `results_summary.json` (mean Â± SD across folds).

In [11]:
results = {
    'run': RUN_TAG,
    'fold': FOLD, 'seed': SEED, 'n_splits': n_splits,
    'config': {'max_len': MAX_LEN, 'n_epochs_max': N_EPOCHS, 'epochs_trained': len(history['val_loss']),
               'batch_size': BATCH_SIZE, 'lr': LR, 'unfrozen_layers': UNFREEZE_LAST_N,
               'imbalance': 'WeightedRandomSampler only', 'model_selection': 'val macro-F1',
               'tokenization': 'scgpt Preprocessor(binning) + tokenize_and_pad_batch(append_cls=True)'},
    'split': {'train_donors': sorted(tr_d), 'val_donors': sorted(va_d), 'test_donors': sorted(te_d),
              'n_train_cells': int(len(train_idx)), 'n_val_cells': int(len(val_idx)),
              'n_test_cells': int(len(test_idx))},
    'majority': majority_metrics,
    'frozen_scgpt_probe': frozen_probe_metrics,
    'finetuned_scgpt': finetuned_metrics,
    'history': history,
}
out = f'results/scgpt_{RUN_TAG}.json'
with open(out, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved {out}')

Saved results/scgpt_fold2_seed7.json
